In [0]:
%sql
SELECT
    DISTINCT status_label
FROM
    crm_ingestion.silver.leads_processed
WHERE   
    status_label = 'Coaching Client'


In [0]:
%sql
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT id) as unique_leads
FROM crm_ingestion.silver.leads_processed
WHERE status_label = 'Coaching Client'

In [0]:
%sql 
WITH numbered_id AS (
    SELECT
        id,
        status_label,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
    FROM
        crm_ingestion.silver.leads_processed
    WHERE
        status_label = 'Coaching Client'
)

SELECT
    *
FROM
    numbered_id
WHERE
    rn = 1
LIMIT 
    10;

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.silver.leads_processed_custom

In [0]:
%sql
CREATE OR REPLACE VIEW crm_ingestion.gold.view_customer_health_base AS (
WITH leads_processed_deduped AS (
    SELECT
    *
    FROM (
        SELECT
            id,
            status_label,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed
        WHERE
            status_label = 'Coaching Client'
        ) 
        WHERE
            rn = 1
)
,
leads_processed_custom_deduped AS (
    SELECT
        *
    FROM (
        SELECT
            leads_processed_id,
            SENTIMENTS_LAST_30_DAYS,
            DAYS_SINCE_LAST_EMAIL, -- — email recency
            DAYS_SINCE_LAST_LOGIN, -- — platform recency
            DAYS_SINCE_LAST_MEETING, -- — meeting recency
            DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, -- — Slack recency (student)
            DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER, -- — Slack recency (team)
            SLACK_ENGAGED_FLAG,
            PLATFORM_ENGAGED_FLAG,
            MEETING_ENGAGED_FLAG,
            TOTAL_MISSED_CALLS, -- — missed call count
            ENGAGEMENT_PATTERN, -- — pre-calculated pattern
            FINAL_SCORE, -- — pre-calculated score
            HEALTH_BAND, -- — pre-calculated band
            CSM, -- — customer success manager
            Contracted_Value, -- — sales context
            Tier, -- — account info
            Avatar, -- profile image
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
    )
    WHERE
        rn = 1
)
SELECT
    l.id,
    l.status_label,
    l.date_updated,
    l.bronze_insert_date,
    NULLIF(c.SENTIMENTS_LAST_30_DAYS, '\\N') AS SENTIMENTS_LAST_30_DAYS,
    NULLIF(c.DAYS_SINCE_LAST_EMAIL, '\\N') AS DAYS_SINCE_LAST_EMAIL,
    NULLIF(c.DAYS_SINCE_LAST_LOGIN, '\\N') AS DAYS_SINCE_LAST_LOGIN,
    NULLIF(c.DAYS_SINCE_LAST_MEETING, '\\N') AS DAYS_SINCE_LAST_MEETING,
    NULLIF(c.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, '\\N') AS DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
    NULLIF(c.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER, '\\N') AS DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    NULLIF(c.SLACK_ENGAGED_FLAG, '\\N') AS SLACK_ENGAGED_FLAG,
    NULLIF(c.PLATFORM_ENGAGED_FLAG, '\\N') AS PLATFORM_ENGAGED_FLAG,
    NULLIF(c.MEETING_ENGAGED_FLAG, '\\N') AS MEETING_ENGAGED_FLAG,
    NULLIF(c.TOTAL_MISSED_CALLS, '\\N') AS TOTAL_MISSED_CALLS,
    NULLIF(c.ENGAGEMENT_PATTERN, '\\N') AS ENGAGEMENT_PATTERN,
    NULLIF(c.FINAL_SCORE, '\\N') AS FINAL_SCORE,
    NULLIF(c.HEALTH_BAND, '\\N') AS HEALTH_BAND,
    NULLIF(c.CSM, '\\N') AS CSM,
    NULLIF(c.Contracted_Value, '\\N') AS Contracted_Value,
    NULLIF(c.Tier, '\\N') AS Tier,
    NULLIF(c.Avatar, '\\N') AS Avatar
FROM
    leads_processed_deduped l
LEFT JOIN
    leads_processed_custom_deduped c
ON
    l.id = c.leads_processed_id
)


In [0]:
%sql
SELECT COUNT(*) FROM crm_ingestion.gold.view_customer_health_base

In [0]:
display(spark.table('crm_ingestion.silver.lead_activites_processed').limit(30))

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.silver.lead_activites_processed

In [0]:
%sql
SELECT DISTINCT status FROM crm_ingestion.silver.lead_activites_processed WHERE type = 'Email'

In [0]:
%sql
WITH deduped_id_activities_filtered_email AS (
    SELECT
        a.id,
        a.lead_id, 
        a.type,
        a.status,
        a.activity_at,
        b.body_text,
        ROW_NUMBER() OVER (PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed a
    LEFT JOIN
        crm_ingestion.silver.lead_activites_email_body b
    ON
        a.id = b.activity_id
    WHERE 
        a.type = 'Email'
    AND a.status = 'inbox'
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%A new event has been scheduled.%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%is requesting access to the following folder%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%requests access to an item%')
),
latest_email_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_email
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_email
FROM
    latest_email_per_lead_id
WHERE
    lead_id_rn = 1

In [0]:
display(spark.table('crm_ingestion.silver.lead_activites_email_body').limit(10))

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.silver.lead_activites_email_body

In [0]:
%sql
WITH deduped_id_activities_filtered_text AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY activity_id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_email_body
    WHERE
        body_text ILIKE '%A new event has been scheduled.%'
),
latest_text_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_text
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_meeting
FROM
    latest_text_per_lead_id
WHERE
    lead_id_rn = 1  

In [0]:
%sql
    
WITH deduped_id AS (
    SELECT
        id,
        lead_id,
        status,
        activity_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Call'
        AND direction = 'inbound'
        AND disposition = 'no-answer'
),
deduped_calls AS (
    SELECT *
    FROM deduped_id
    WHERE id_rn = 1
)
SELECT
    lead_id,
    COUNT(*) AS missed_calls_14_days
FROM
    deduped_calls
WHERE
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) <= 14
GROUP BY
    lead_id

In [0]:
%sql
WITH deduped_id AS (
    SELECT
        id,
        lead_id,
        status,
        starts_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Meeting'
        AND status = 'upcoming'
        AND DATE(starts_at) > CURRENT_DATE
),
earliest_upcoming_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY starts_at ASC) AS lead_id_rn
    FROM
        deduped_id
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(DATE(starts_at), CURRENT_DATE) AS days_until_next_meeting
FROM
    earliest_upcoming_per_lead
WHERE
    lead_id_rn = 1